# Publication Figures — One-Button Reproduction

Regenerates every committed figure and LaTeX table in the publication bundle
(`results/figures/`, `results/tables/`) via `scripts/generate_all_figures.py`.

> Last verified: 2026-06-19 · Estimated runtime: **< 1 min** (CPU-only)

Use this notebook when you only need the four paper figures and summary tables,
without running the full benchmark or real-dataset suites.

In [ ]:
# ── Install quantum-oracle-sketching (idempotent) ──────────────────────────
import subprocess, sys, os
_REPO = "https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git"
_CLONE_DIR = "./quantum_oracle_sketching"
if not os.path.isdir(_CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-q", _REPO, _CLONE_DIR],
                   check=True)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e",
     f"{_CLONE_DIR}[dev,noise,kernel]"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Install failed:\n{result.stderr}")
import importlib, importlib.util, site
importlib.invalidate_caches()
site.main()
_SRC_DIR = os.path.abspath(os.path.join(_CLONE_DIR, "src"))
if importlib.util.find_spec("qos") is None and os.path.isdir(_SRC_DIR):
    if _SRC_DIR not in sys.path:
        sys.path.insert(0, _SRC_DIR)
    importlib.invalidate_caches()
_spec = importlib.util.find_spec("qos")
if _spec is None:
    raise ImportError(
        "qos is still not importable after install.\n"
        "Runtime > Disconnect and delete runtime, then Run All."
    )
print("✅ quantum-oracle-sketching installed.")
print(f"✅ qos found at: {_spec.origin}")

In [ ]:
# ── Resolve repo root (Colab clone vs local editable install) ─────────────
import os, sys
from pathlib import Path

def _find_repo_root() -> Path:
    candidates = [
        Path("./quantum_oracle_sketching"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for c in candidates:
        if (c / "scripts" / "generate_all_figures.py").is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Cannot locate scripts/generate_all_figures.py. "
        "Run the install cell first or open from the repo root."
    )

REPO_ROOT = _find_repo_root()
FIG_DIR = REPO_ROOT / "results" / "figures"
TBL_DIR = REPO_ROOT / "results" / "tables"
print(f"Repo root : {REPO_ROOT}")
print(f"Figures   : {FIG_DIR}")
print(f"Tables    : {TBL_DIR}")

In [ ]:
# ── Regenerate all publication figures and tables ─────────────────────────
import subprocess, sys, os, time

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
script = REPO_ROOT / "scripts" / "generate_all_figures.py"

t0 = time.time()
proc = subprocess.run(
    [sys.executable, str(script)],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
elapsed = time.time() - t0
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"generate_all_figures.py failed (rc={proc.returncode})")
print(f"\n✅ All figures regenerated in {elapsed:.1f}s")

In [ ]:
# ── Verify publication bundle (CI gate) ───────────────────────────────────
import subprocess, sys, os

verify = REPO_ROOT / "scripts" / "verify_publication_bundle.py"
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
proc = subprocess.run(
    [sys.executable, str(verify)],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Publication bundle verification failed")
print("✅ Publication bundle verified.")

## Publication Figures

Four figure pairs (PNG + PDF) committed to `results/figures/`.

In [ ]:
from IPython.display import display, Image, Markdown
from pathlib import Path

FIGURES = [
    ("tvd_convergence.png", "TVD vs. sample budget $M$ (cold uniform sketch)"),
    ("sample_complexity.png", "Smallest $M^\star$ achieving TVD $< 0.10$ vs. $N$"),
    ("noise_robustness.png", "Depolarizing-noise robustness (TVD vs. $\eta$)"),
    ("circuit_depth.png", "Circuit-depth crossover (samples vs. depth)"),
]

for fname, caption in FIGURES:
    path = FIG_DIR / fname
    if not path.is_file():
        raise FileNotFoundError(f"Missing figure: {path}")
    display(Markdown(f"### {caption}"))
    display(Image(filename=str(path), width=700))

## Summary Tables

LaTeX tables under `results/tables/` (included in `paper/sections/experiments.tex`).

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

for tex in sorted(TBL_DIR.glob("table_*.tex")):
    display(Markdown(f"### `{tex.name}`"))
    print(tex.read_text())